<a href="https://colab.research.google.com/github/saiKelkar/Property_Splats/blob/main/02_full_quality_room.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== SESSION START =====
PROJECT = "mipnerf360_room"
ITERATIONS = 30000

from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob, psutil
SPLATS = "/content/drive/MyDrive/splats"
DRIVE_DIR = f"{SPLATS}/{PROJECT}"
WORK = f"/content/{PROJECT}"
os.makedirs(WORK, exist_ok=True)

ram_gb = psutil.virtual_memory().total / 1e9
JOBS = max(1, min(os.cpu_count(), int(ram_gb // 6)))
print(f"RAM: {ram_gb:.0f} GB | CPUs: {os.cpu_count()} | compile jobs: {JOBS}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

!apt-get -qq update > /dev/null && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio

wheels = sorted(glob.glob(f"{SPLATS}/wheels/gsplat-*.whl"))
if wheels:
    !pip -q install --no-deps --force-reinstall "{wheels[-1]}"
    print("Installed pre-built gsplat from Drive")
else:
    print("No gsplat wheel in Drive: gsplat will compile on first use (slow)")

if os.path.exists(f"{DRIVE_DIR}/room.zip") and not os.path.exists(f"{WORK}/room"):
    shutil.unpack_archive(f"{DRIVE_DIR}/room.zip", WORK)
    print("Restored dataset from Drive")

!python -c "from gsplat import csrc; print('gsplat ready (pre-built):', csrc is not None)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAM: 55 GB | CPUs: 8 | compile jobs: 8
name, memory.total [MiB]
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Restored compiled gsplat from Drive
gsplat CUDA ready


In [ ]:
!MAX_JOBS={JOBS} ns-train splatfacto --output-dir {WORK}/outputs --vis tensorboard --max-num-iterations {ITERATIONS} colmap --data {WORK}/room --colmap-path sparse/0 --downscale-factor 2 --downscale-rounding-mode ceil > {WORK}/train_log.txt 2>&1; echo "EXIT CODE: $?"
!tail -n 25 {WORK}/train_log.txt

In [3]:
import glob, shutil, os
configs = [c for c in glob.glob(f"{WORK}/outputs/**/config.yml", recursive=True)
           if glob.glob(os.path.join(os.path.dirname(c), "nerfstudio_models", "*.ckpt"))]
config = sorted(configs)[-1]
print("Using:", config)
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat --load-config {config} --output-dir "{DRIVE_DIR}/export_30k"
!ls -lh "{DRIVE_DIR}/export_30k"

Using: /content/mipnerf360_room/outputs/unnamed/splatfacto/2026-11-20_073841/config.yml
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795163900.929507   35453 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Ple

In [4]:
run_dir = os.path.dirname(config)
shutil.copytree(run_dir, f"{DRIVE_DIR}/runs/{os.path.basename(run_dir)}", dirs_exist_ok=True)
shutil.copy(f"{WORK}/train_log.txt", f"{DRIVE_DIR}/train_log_30k.txt")
print("Saved run + log to Drive")

Saved run + log to Drive


In [5]:
!pip -q install plyfile
import numpy as np, plotly.graph_objects as go, glob
from plyfile import PlyData

ply_path = glob.glob(f"{DRIVE_DIR}/export_30k/*.ply")[0]
v = PlyData.read(ply_path)["vertex"].data

xyz = np.stack([v["x"], v["y"], v["z"]], axis=1)
SH_C0 = 0.28209479177387814
rgb = np.clip(0.5 + SH_C0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], axis=1), 0, 1)
opacity = 1 / (1 + np.exp(-v["opacity"]))
print(f"{len(xyz):,} splats in the file")

keep = opacity > 0.5
lo, hi = np.percentile(xyz[keep], [1, 99], axis=0)
keep &= np.all((xyz > lo) & (xyz < hi), axis=1)
idx = np.random.default_rng(0).choice(np.flatnonzero(keep), size=min(150_000, keep.sum()), replace=False)

colors = [f"rgb({r},{g},{b})" for r, g, b in (rgb[idx] * 255).astype(int)]
fig = go.Figure(go.Scatter3d(x=xyz[idx, 0], y=xyz[idx, 1], z=xyz[idx, 2], mode="markers",
                             marker=dict(size=1, color=colors)))
fig.update_layout(scene=dict(aspectmode="data", bgcolor="black"), height=700, margin=dict(l=0, r=0, t=0, b=0))
fig.show()

553,477 splats in the file


In [6]:
!mkdir -p "{SPLATS}/wheels"
!nohup env TORCH_CUDA_ARCH_LIST="7.5;8.0;8.9" MAX_JOBS={JOBS} pip wheel --no-build-isolation --no-deps --no-binary gsplat gsplat==1.4.0 -w "{SPLATS}/wheels" > "{SPLATS}/wheel_build_log.txt" 2>&1 &
print("Wheel build started in the background")

Wheel build started in the background


In [15]:
!tail -n 5 "{SPLATS}/wheel_build_log.txt"
!ls -lh "{SPLATS}/wheels"

  copying gsplat/cuda/csrc/third_party/glm/util/glm.natvis -> build/lib.linux-x86_64-cpython-313/gsplat/cuda/csrc/third_party/glm/util
  copying gsplat/cuda/csrc/third_party/glm/util/usertype.dat -> build/lib.linux-x86_64-cpython-313/gsplat/cuda/csrc/third_party/glm/util
  running build_ext
  building 'gsplat.csrc' extension
  creating /tmp/pip-req-build-bc31_qee/build/temp.linux-x86_64-cpython-313/gsplat/cuda/csrc
total 0


In [11]:
!grep -n -i -E "error|fatal|killed|no such file" "{SPLATS}/wheel_build_log.txt" | head -30

7:  error: subprocess-exited-with-error
13:  note: This error originates from a subprocess, and is likely not a problem with pip.
14:  Building wheel for gsplat (pyproject.toml): finished with status 'error'
15:  ERROR: Failed building wheel for gsplat
17:ERROR: Failed to build one or more wheels


In [12]:
!cat "{SPLATS}/wheel_build_log.txt"

  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  error: subprocess-exited-with-error
  
  × Building wheel for gsplat (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for gsplat
Failed to build gsplat
ERROR: Failed to build one or more wheels


In [14]:
!nohup env TORCH_CUDA_ARCH_LIST="7.5;8.0;8.9" MAX_JOBS={JOBS} pip wheel -v --no-build-isolation --no-deps "git+https://github.com/nerfstudio-project/gsplat.git@v1.4.0" -w "{SPLATS}/wheels" > "{SPLATS}/wheel_build_log.txt" 2>&1 &
print("Building from GitHub source (v1.4.0)")

Building from GitHub source (v1.4.0)


In [18]:
!grep -n -E " error|error:|FAILED|Killed" "{SPLATS}/wheel_build_log.txt" | head -15
!echo "compiler processes running:" $(ps aux | grep -c "[n]vcc")
!ls -lh "{SPLATS}/wheels"

compiler processes running: 18
total 0


In [19]:
!pip -q install --no-deps --force-reinstall "{SPLATS}"/wheels/gsplat-*.whl
!python -c "from gsplat import csrc; print('Pre-built gsplat works:', csrc is not None)"

Pre-built gsplat works: True
